<a href="https://colab.research.google.com/github/anbaluxe/my_colab_learning/blob/main/%D0%A1%D1%82%D1%80%D0%B0%D1%82%D0%B8%D1%84%D0%B8%D1%86%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%BD%D0%B4%D0%BE%D0%BC%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_users = 10_000

df = pd.DataFrame({
    "user_id": np.arange(1, n_users + 1),

    "os": np.random.choice(
        ["Android", "iOS"],
        size=n_users,
        p=[0.7, 0.3]
    )
})

df.head()

,user_id,os
0,1,Android
1,2,iOS
2,3,iOS
3,4,Android
4,5,Android


In [ ]:
df["os"].value_counts(normalize=True)

,proportion
os,
Android,0.7113
iOS,0.2887


In [ ]:
df["rpu"] = np.where(
    df["os"] == "Android",
    np.random.normal(20, 5, n_users),
    np.random.normal(100, 10, n_users)
)

In [ ]:
df.groupby("os")["rpu"].agg(["count", "mean", "var", "std"])

,count,mean,var,std
os,,,,
Android,7113,20.123618,25.252115,5.025148
iOS,2887,99.992981,98.242981,9.911760


Рандомизация по OS

In [ ]:
df['group'] = 'Control' # Заполняем все строки Control

test_idx = (
    df.groupby('os')
      .sample(frac=0.5, random_state=42)
      .index
)

'''
  1. Разделяем по os
  2. Случайная рандомизация внутри каждой группы frac=0.5 50%
  3. Получаемя index-сы пользоваателей
'''

df.loc[test_idx, 'group'] = 'Test'

In [ ]:
pd.crosstab(
    df["os"],
    df["group"]
)

group,Control,Test
os,,
Android,3557,3556
iOS,1443,1444


In [ ]:
pd.crosstab(
    df["os"],
    df["group"],
    normalize="index"
)

group,Control,Test
os,,
Android,0.500070,0.499930
iOS,0.499827,0.500173


In [ ]:
pd.crosstab(
    df["group"],
    df["os"],
    normalize="index"
)

os,Android,iOS
group,,
Control,0.7114,0.2886
Test,0.7112,0.2888


In [ ]:
df["rpu_test"] = df["rpu"]

df.loc[df["group"] == "Test", "rpu_test"] *= 1.05

In [ ]:
strat = df.groupby("group")["rpu_test"].agg(
    ["count", "mean", "var", "std"]
)

print(strat)

strat["var_mean"] = strat['var'] / strat['count']

se = (
    strat.loc["Test", "var_mean"]
    + strat.loc["Control", "var_mean"]
) ** 0.5

delta_srat = strat.loc["Test", "mean"] - strat.loc["Control", "mean"]

print('SE', se)
print('delta', delta_srat)

t_stat_strat = delta_srat / se

print(t_stat_strat)

ci_left_strat = delta_srat - 1.96 * se
ci_rigth_strat = delta_srat + 1.96 * se

print(ci_left_strat, ci_rigth_strat)

         count       mean          var        std
group                                            
Control   5000  43.043866  1342.557604  36.640928
Test      5000  45.485938  1510.979975  38.871326
SE 0.7554518619672222
delta 2.4420714363207097
3.2325970181097614
0.961385786864954 3.9227570857764653


## Без стратиикации

In [ ]:
df_unstrat = df[["user_id", "os", "rpu"]].copy()

df_unstrat["group"] = "Control"

test_idx = (
    df_unstrat
    .sample(frac=0.5, random_state=42)
    .index
)

df_unstrat.loc[test_idx, "group"] = "Test"

In [ ]:
pd.crosstab(
    df_unstrat["group"],
    df_unstrat["os"]
)

os,Android,iOS
group,,
Control,3535,1465
Test,3578,1422


In [ ]:
pd.crosstab(
    df_unstrat["group"],
    df_unstrat["os"],
    normalize="index"
)

os,Android,iOS
group,,
Control,0.7070,0.2930
Test,0.7156,0.2844


In [ ]:
df_unstrat["rpu_test"] = df_unstrat["rpu"]

df_unstrat.loc[df_unstrat["group"] == "Test", "rpu_test"] *= 1.05

In [ ]:
unstrat = df_unstrat.groupby("group")["rpu_test"].agg(
    ["count", "mean", "var", "std"]
)

In [ ]:
unstrat["var_mean"] = unstrat['var'] / unstrat['count']

unstrat

,count,mean,var,std,var_mean
group,,,,,
Control,5000,43.580847,1369.300774,37.004064,0.273860
Test,5000,44.922108,1481.186645,38.486188,0.296237


In [ ]:
se_unstrat = (
    unstrat.loc["Test", "var_mean"]
    + unstrat.loc["Control", "var_mean"]
) ** 0.5

delta = unstrat.loc["Test", "mean"] - unstrat.loc["Control", "mean"]

print('SE', se_unstrat)
print('delta', delta)

t_stat_unstrat = delta / se_unstrat

print(t_stat_unstrat)

ci_left_unstrat = delta - 1.96 * se_unstrat
ci_rigth_unstrat = delta + 1.96 * se_unstrat

print(ci_left_unstrat, ci_rigth_unstrat)

SE 0.7550480009069714
delta 1.3412614999018118
1.776392359546247
-0.13863258187585226 2.8211555816794758
